# Fine-tune the loans agent model (QLoRA, free Colab T4)

Trains **Llama-3.1-8B-Instruct** (or Qwen2.5-7B-Instruct) on
`data/loans/loans_pakistan_finetune_10k.jsonl` with Unsloth QLoRA, then exports a
**GGUF** you register with **Ollama** so the agent in `app/agents/loans/` uses it
as a drop-in (the agent's prompts already match this dataset's format).

Runtime: **T4 GPU** (Colab free tier) · ~1–2 h for 1 epoch on 10k examples.

Steps: install → upload dataset → format with the chat template → train →
quick eval → export GGUF → `ollama create` → point the agent at it.

In [ ]:
%%capture
!pip install unsloth
# Colab sometimes needs the nightly fix-ups:
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 1 · Upload the dataset
Upload `loans_pakistan_finetune_10k.jsonl` from `data/loans/` in the repo
(or mount Drive and point `DATA_PATH` at it).

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick loans_pakistan_finetune_10k.jsonl
DATA_PATH = next(iter(uploaded))
print('Using', DATA_PATH)

## 2 · Load the base model (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
# Alternative with stronger multilingual (Roman Urdu) coverage:
# MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=47,
)

## 3 · Format the dataset with the model's chat template
The system prompt matches `app/agents/loans/prompts.py::SYSTEM_PROMPT` so
inference-time prompts are in-distribution.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are a senior credit officer at a Pakistani bank running the '
    'responsible instant-lending desk. You read customer relationship data, '
    'eCIB records, and cash-flow signals, then walk through lending decisions '
    'under the bank\'s internal policy. You never recommend unsecured lending '
    'to POOR-band files; you redirect them to secured alternatives. All '
    'amounts are in Pakistani rupees. Be concise, numerate, and direct — a '
    'declined customer should always leave with a workable path.'
)

rows = []
with open(DATA_PATH) as f:
    for line in f:
        d = json.loads(line)
        msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + d['messages']
        rows.append({'text': tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)})

ds = Dataset.from_list(rows).train_test_split(test_size=0.02, seed=47)
print(ds)
print(ds['train'][0]['text'][:600])

## 4 · Train (1 epoch)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch 16
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='no',
        output_dir='outputs',
        seed=47,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        report_to='none',
    ),
)
trainer.train()

## 5 · Smoke-test before exporting

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = '''CUSTOMER PROFILE
- Name: Test Customer, 35
- Bank: Habib Bank Limited (HBL), Lahore
- Employment: Careem captain (gig)
- Verified net monthly income: Rs 90,000
- Account age: 5.0 years
- Salary/inflow months (last 12): 12/12
- Average balance (6-month): Rs 120,000
- Previous loan: fully repaid, never 30+ days late
- Cheque/DD bounce in last 12 months: no
- eCIB 90+ DPD in last 24 months: no
- eCIB write-off/litigation flag: no
- Existing monthly obligations: none
- Days below Rs 5,000 in the last 30 days: 18

Is this customer running low on money? Read the signals.'''

msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': test_prompt}]
inputs = tokenizer.apply_chat_template(
    msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.3)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 6 · Export GGUF for Ollama
`q4_k_m` is the right size/quality trade-off for local CPU/GPU serving.

In [ ]:
model.save_pretrained_gguf('loans-agent-gguf', tokenizer, quantization_method='q4_k_m')
# Download the .gguf file (or copy it to Drive — it is ~5 GB):
import glob; print(glob.glob('loans-agent-gguf/*.gguf'))

## 7 · Register with Ollama on your local machine

```bash
cat > Modelfile <<'EOF'
FROM ./loans-agent-q4_k_m.gguf
PARAMETER temperature 0.3
SYSTEM You are a senior credit officer at a Pakistani bank running the responsible instant-lending desk. You read customer relationship data, eCIB records, and cash-flow signals, then walk through lending decisions under the bank's internal policy. You never recommend unsecured lending to POOR-band files; you redirect them to secured alternatives. All amounts are in Pakistani rupees. Be concise, numerate, and direct — a declined customer should always leave with a workable path.
EOF
ollama create loans-agent -f Modelfile
```

Then point the agent at it — no code changes needed:

```bash
export LOAN_LLM_MODEL=loans-agent
export LOAN_FEWSHOT=0   # fine-tuned model no longer needs few-shot examples
python scripts/loan_demo.py --walkthrough
```